# SVM checkpoint sanity: legacy `.pkl`+zscores vs Pipeline `.joblib`

Runs `scripts/data_evaluation/run_model_inference.py` in `--svm-only` mode twice:

1. **Legacy** — `svm_qsar12_model.pkl` + `svm_qsar12_zscores.txt`
2. **Pipeline** — `StandardScaler` + `SVC` joblib (built from the legacy pair if missing)

Then compares `SVM_pred`, `SVM_prob_AMP`, and `SVM_distance` / `SVM_hyperplane_distance`.

Defaults come from `configs/compare_models.json` and the CPPs test workspace.

In [1]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

here = Path.cwd().resolve()
repo = here.parent if here.name == "notebooks" else here
ROOT = repo / "sequence_to_svm_minimal"
if not (ROOT / "configs" / "compare_models.json").is_file():
    if (here / "configs" / "compare_models.json").is_file():
        ROOT = here
    elif (repo / "configs" / "compare_models.json").is_file():
        ROOT = repo

sys.path.insert(0, str(ROOT))
from configs.load_config import load_compare_models_config

cfg = load_compare_models_config()
WORKSPACE = ROOT / "data" / "test" / "CPPs" / "generated"
INFERENCE = ROOT / "scripts" / "data_evaluation" / "run_model_inference.py"
CONVERT = ROOT / "scripts" / "convert_svm_pkl_zscores_to_pipeline.py"

SVM_PKL = Path(cfg["svm_pkl"])
SVM_Z = Path(cfg["svm_z_file"])
SVM_DESC = Path(cfg["svm_descriptor_csv"])
if WORKSPACE.is_dir() and (WORKSPACE / "qsar12_descriptors.csv").is_file():
    SVM_DESC = WORKSPACE / "qsar12_descriptors.csv"

OUT_DIR = ROOT / "results" / "svm_sanity"
PIPELINE = OUT_DIR / "svm_qsar12_pipeline.joblib"
LEGACY_CSV = OUT_DIR / "svm_only_legacy.csv"
PIPELINE_CSV = OUT_DIR / "svm_only_pipeline.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

assert INFERENCE.is_file(), INFERENCE
assert WORKSPACE.is_dir(), WORKSPACE
assert SVM_PKL.is_file(), SVM_PKL
assert SVM_Z.is_file(), SVM_Z
assert SVM_DESC.is_file(), SVM_DESC

print("ROOT", ROOT)
print("workspace", WORKSPACE)
print("legacy", SVM_PKL.name, "+", SVM_Z.name)
print("descriptors", SVM_DESC)

ROOT /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal
workspace /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/data/test/CPPs/generated
legacy svm_qsar12_model.pkl + svm_qsar12_zscores.txt
descriptors /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/data/test/CPPs/generated/qsar12_descriptors.csv


## 1. Build Pipeline joblib from legacy artifacts (no retrain)

In [2]:
cmd_convert = [
    sys.executable,
    str(CONVERT),
    "--svm_pkl", str(SVM_PKL),
    "--svm_z_file", str(SVM_Z),
    "--out", str(PIPELINE),
]
print(" ".join(cmd_convert))
subprocess.run(cmd_convert, check=True, cwd=str(ROOT))
assert PIPELINE.is_file(), PIPELINE
print("pipeline", PIPELINE)

/home/bioin/miniconda3/envs/gnn/bin/python /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/scripts/convert_svm_pkl_zscores_to_pipeline.py --svm_pkl /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/checkpoints/latest/svm_qsar12_model.pkl --svm_z_file /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/checkpoints/latest/svm_qsar12_zscores.txt --out /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/results/svm_sanity/svm_qsar12_pipeline.joblib


Wrote Pipeline (StandardScaler + SVC): /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/results/svm_sanity/svm_qsar12_pipeline.joblib
  features (12): netCharge, FC, LW, DP, NK, AE, pcMK, _SolventAccessibilityD1025...
Inference: pipe.predict(X_raw) / predict_proba / decision_function on unscaled columns.
Note: compare_model_predictions.py still expects the legacy .pkl + zscores pair;
      update loaders separately if you want to consume this pipeline file.
pipeline /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/results/svm_sanity/svm_qsar12_pipeline.joblib


## 2. Run `run_model_inference.py --svm-only` (legacy + pipeline)

In [3]:
def run_svm_only(
    *,
    output_csv: Path,
    svm_pkl: Path | None = None,
    svm_z: Path | None = None,
    svm_pipeline: Path | None = None,
) -> Path:
    cmd = [
        sys.executable,
        str(INFERENCE),
        str(WORKSPACE),
        "--svm-only",
        "--svm-descriptor-csv", str(SVM_DESC),
        "--output-csv", str(output_csv),
    ]
    if svm_pipeline is not None:
        cmd += ["--svm-pipeline", str(svm_pipeline)]
    else:
        assert svm_pkl is not None and svm_z is not None
        cmd += ["--svm-pkl", str(svm_pkl), "--svm-z-file", str(svm_z)]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True, cwd=str(ROOT))
    return output_csv

run_svm_only(output_csv=LEGACY_CSV, svm_pkl=SVM_PKL, svm_z=SVM_Z)
run_svm_only(output_csv=PIPELINE_CSV, svm_pipeline=PIPELINE)
print("wrote", LEGACY_CSV)
print("wrote", PIPELINE_CSV)

/home/bioin/miniconda3/envs/gnn/bin/python /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/scripts/data_evaluation/run_model_inference.py /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/data/test/CPPs/generated --svm-only --svm-descriptor-csv /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/data/test/CPPs/generated/qsar12_descriptors.csv --output-csv /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/results/svm_sanity/svm_only_legacy.csv --svm-pkl /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/checkpoints/latest/svm_qsar12_model.pkl --svm-z-file /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/checkpoints/latest/svm_qsar12_zscores.txt


Log: /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/data/test/CPPs/generated/logs/run_model_inference_20260812_115336.log
Command: /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/scripts/data_evaluation/run_model_inference.py /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/data/test/CPPs/generated --svm-only --svm-descriptor-csv /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/data/test/CPPs/generated/qsar12_descriptors.csv --output-csv /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/results/svm_sanity/svm_only_legacy.csv --svm-pkl /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/sequence_to_svm_minimal/checkpoints/latest/svm_qsar12_model.pkl --svm-z-file /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Propertie

## 3. Compare predictions

In [4]:
legacy = pd.read_csv(LEGACY_CSV)
pipe = pd.read_csv(PIPELINE_CSV)

key = "peptide_id"
score_cols = ["SVM_pred", "SVM_prob_AMP", "SVM_distance", "SVM_hyperplane_distance"]
for col in score_cols:
    assert col in legacy.columns, f"missing {col} in legacy"
    assert col in pipe.columns, f"missing {col} in pipeline"

merged = legacy[[key] + score_cols].merge(
    pipe[[key] + score_cols],
    on=key,
    how="inner",
    suffixes=("_legacy", "_pipeline"),
)
assert len(merged) > 0, "no overlapping peptide_ids"

pred_match = (merged["SVM_pred_legacy"] == merged["SVM_pred_pipeline"]).all()
prob_max_abs = float(
    np.nanmax(np.abs(merged["SVM_prob_AMP_legacy"] - merged["SVM_prob_AMP_pipeline"]))
)
dist_max_abs = float(
    np.nanmax(np.abs(merged["SVM_distance_legacy"] - merged["SVM_distance_pipeline"]))
)

print(f"n peptides compared: {len(merged):,}")
print(f"SVM_pred exact match: {bool(pred_match)}")
print(f"max |Δ SVM_prob_AMP|: {prob_max_abs:.3e}")
print(f"max |Δ SVM_distance|: {dist_max_abs:.3e}")

ATOL_PROB = 1e-10
ATOL_DIST = 1e-10
ok = pred_match and prob_max_abs <= ATOL_PROB and dist_max_abs <= ATOL_DIST
if not ok:
    bad = merged[
        (merged["SVM_pred_legacy"] != merged["SVM_pred_pipeline"])
        | (np.abs(merged["SVM_prob_AMP_legacy"] - merged["SVM_prob_AMP_pipeline"]) > ATOL_PROB)
        | (np.abs(merged["SVM_distance_legacy"] - merged["SVM_distance_pipeline"]) > ATOL_DIST)
    ]
    display(bad.head(20))
    raise AssertionError(
        f"Legacy vs Pipeline mismatch (n_bad={len(bad)}, "
        f"prob_max_abs={prob_max_abs}, dist_max_abs={dist_max_abs})"
    )

print("PASS: legacy and Pipeline SVM outputs match within tolerance.")
merged.head()

n peptides compared: 708
SVM_pred exact match: True
max |Δ SVM_prob_AMP|: 0.000e+00
max |Δ SVM_distance|: 0.000e+00
PASS: legacy and Pipeline SVM outputs match within tolerance.


,peptide_id,SVM_pred_legacy,SVM_prob_AMP_legacy,SVM_distance_legacy,SVM_hyperplane_distance_legacy,SVM_pred_pipeline,SVM_prob_AMP_pipeline,SVM_distance_pipeline,SVM_hyperplane_distance_pipeline
0,SEQ_1,1,0.961327,1.923250,1.923250,1,0.961327,1.923250,1.923250
1,SEQ_10,1,0.948877,1.752855,1.752855,1,0.948877,1.752855,1.752855
2,SEQ_100,0,0.021292,-2.189088,-2.189088,0,0.021292,-2.189088,-2.189088
3,SEQ_101,1,0.601706,0.290115,0.290115,1,0.601706,0.290115,0.290115
4,SEQ_102,1,0.817882,0.926769,0.926769,1,0.817882,0.926769,0.926769
